## Giải thích cơ chế Semantic Chunking
 
**Semantic chunking** là phương pháp tách văn bản thành các đoạn nhỏ (chunk) dựa trên ý nghĩa, chủ đề hoặc cấu trúc ngữ nghĩa thay vì chỉ dựa vào ký tự phân tách như truyền thống.
- Thay vì cắt theo số ký tự, dòng, hoặc từ, semantic chunking sử dụng các mô hình NLP (ví dụ: embedding, topic modeling, sentence boundary detection) để xác định ranh giới tự nhiên giữa các ý hoặc chủ đề.
- Mục tiêu là giữ nguyên ngữ cảnh, ý nghĩa và logic của từng đoạn, giúp các tác vụ như truy vấn, embedding, RAG hiệu quả hơn.
 
**Ưu điểm:**
- Các chunk giữ được ý nghĩa trọn vẹn, không bị cắt ngang chủ đề.
- Phù hợp cho các bài toán cần hiểu sâu về ngữ nghĩa, như semantic search, QA, summarization.

**Nhược điểm:**
- Hiệu quả phụ thuộc vào việc chọn ngưỡng (threshold) tương đồng phù hợp, ngưỡng này có thể thay đổi tùy thuộc vào loại tài liệu và mô hình embedding.
- Tính toán embedding và độ tương đồng có thể tốn kém hơn fixed-size chunking (tốn tiền API và tài nguyên 😣).
- Các chunk sẽ có size chênh lệch khá lớn.  

 
**Ví dụ:**
- Tách văn bản thành các đoạn theo từng chủ đề, từng câu chuyện, hoặc từng ý chính.
- Sử dụng mô hình embedding để đo độ tương đồng giữa các câu, từ đó xác định điểm cắt hợp lý.
 
**Ứng dụng:**
- Semantic chunking thường dùng trong các hệ thống RAG, chatbot, search engine, hoặc khi cần lưu trữ tri thức có cấu trúc logic rõ ràng.

In [8]:
text = """
MARLEY'S GHOST

Marley was dead, to begin with. There is no doubt whatever about that. The register of his burial was signed by the clergyman, the clerk, the undertaker, and the chief mourner. Scrooge signed it. And Scrooge's name was good upon 'Change for anything he chose to put his hand to. Old Marley was as dead as a door-nail.

Mind! I don't mean to say that I know of my own knowledge, what there is particularly dead about a door-nail. I might have been inclined, myself, to regard a coffin-nail as the deadest piece of ironmongery in the trade. But the wisdom of our ancestors is in the simile; and my unhallowed hands shall not disturb it, or the country's done for. You will, therefore, permit me to repeat, emphatically, that Marley was as dead as a door-nail.

Scrooge knew he was dead? Of course he did. How could it be otherwise? Scrooge and he were partners for I don't know how many years. Scrooge was his sole executor, his sole administrator, his sole assign, his sole particularly dead about a door-nail. I might have been inclined, myself, to regard a coffin-nail as the deadest piece of ironmongery in the trade. But the wisdom of our ancestors is in the simile; and my unhallowed hands shall not disturb it, or the country's done for. You will, therefore, permit me to repeat, emphatically, that Marley was as dead as a door-nail.



"""

# Define Semantic Splitter

In [7]:
from llama_index.core.node_parser import (
    SentenceSplitter,
    SemanticSplitterNodeParser,
)
from llama_index.embeddings.openai import OpenAIEmbedding

import os
from dotenv import load_dotenv
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")


In [10]:
#define doc
from llama_index.core import Document

docs  = [Document(text= text)]

### Giải thích tham số semantic chunking
 
- **buffer_size**: Số lượng câu sẽ gộp thành 1 nhóm trước khi thực hiện semantic chunking.
    - Mặc định = 1: mỗi chunk là một câu.
    - Lớn hơn 1: mỗi chunk là một đoạn gồm nhiều câu.
 
- **breakpoint_percentile_threshold**: Giá trị kiểm soát việc chia chunk dựa trên độ khác biệt ngữ nghĩa giữa nhóm câu hiện tại và câu tiếp theo.
    - Giá trị càng lớn → yêu cầu sự khác biệt ngữ nghĩa càng lớn để tách (chunk sẽ lớn, ít chunk được tạo ra).
    - Giá trị càng nhỏ → chỉ cần khác biệt nhỏ đã tách (chunk sẽ nhỏ, nhiều chunk hơn).
    - Hiểu đơn giản: đây là cosine dissimilarity (ngược với cosine similarity).
        - = 0: giống nhau hoàn toàn.
        - = 1: khác nhau hoàn toàn.
 
**Lưu ý:**
- Điều chỉnh hai tham số này giúp kiểm soát kích thước và ý nghĩa của các chunk, phù hợp với từng loại văn bản hoặc mục đích sử dụng.

In [20]:
embed_model = OpenAIEmbedding()
splitter = SemanticSplitterNodeParser(
    buffer_size=1, 
    breakpoint_percentile_threshold=95, 
    embed_model=embed_model
)
nodes = splitter.get_nodes_from_documents(docs)

In [21]:
from pprint import pprint
for i, node in enumerate(nodes):
    print(f"Node {i+1}:")
    pprint(node.get_text())
    print("-" * 40)

Node 1:
('\n'
 "MARLEY'S GHOST\n"
 '\n'
 'Marley was dead, to begin with. There is no doubt whatever about that. The '
 'register of his burial was signed by the clergyman, the clerk, the '
 "undertaker, and the chief mourner. Scrooge signed it. And Scrooge's name was "
 "good upon 'Change for anything he chose to put his hand to. Old Marley was "
 'as dead as a door-nail.\n'
 '\n')
----------------------------------------
Node 2:
("Mind! I don't mean to say that I know of my own knowledge, what there is "
 'particularly dead about a door-nail. I might have been inclined, myself, to '
 'regard a coffin-nail as the deadest piece of ironmongery in the trade. But '
 'the wisdom of our ancestors is in the simile; and my unhallowed hands shall '
 "not disturb it, or the country's done for. You will, therefore, permit me to "
 'repeat, emphatically, that Marley was as dead as a door-nail.\n'
 '\n'
 'Scrooge knew he was dead? Of course he did. How could it be otherwise? '
 "Scrooge and he wer

Có thể thấy rằng nếu tôi để `breakpoint_percentile_threshold=95` thì các chunk sẽ rất dài, do cái threshold để cắt ra khá là cao, ( độ **không tương quan** phải hơn **95** thì mới cắt)

Nhưng nếu tôi để `threshold` nhỏ hơn một chút (khoảng 75) sẽ chia thành nhiều chunk hơn với nội dung bị chia nhỏ ra hơn.

In [22]:
splitter = SemanticSplitterNodeParser(
    buffer_size=1, 
    breakpoint_percentile_threshold=70, 
    embed_model=embed_model
)
nodes = splitter.get_nodes_from_documents(docs)


for i, node in enumerate(nodes):
    print(f"Node {i+1}:")
    pprint(node.get_text())
    print("-" * 40)

Node 1:
('\n'
 "MARLEY'S GHOST\n"
 '\n'
 'Marley was dead, to begin with. There is no doubt whatever about that. ')
----------------------------------------
Node 2:
('The register of his burial was signed by the clergyman, the clerk, the '
 "undertaker, and the chief mourner. Scrooge signed it. And Scrooge's name was "
 "good upon 'Change for anything he chose to put his hand to. Old Marley was "
 'as dead as a door-nail.\n'
 '\n')
----------------------------------------
Node 3:
'Mind! '
----------------------------------------
Node 4:
("I don't mean to say that I know of my own knowledge, what there is "
 'particularly dead about a door-nail. I might have been inclined, myself, to '
 'regard a coffin-nail as the deadest piece of ironmongery in the trade. But '
 'the wisdom of our ancestors is in the simile; and my unhallowed hands shall '
 "not disturb it, or the country's done for. You will, therefore, permit me to "
 'repeat, emphatically, that Marley was as dead as a door-nail.\n'

### Lưu ý khi tối ưu tham số semantic chunking
 
- Tùy vào cài đặt `breakpoint_percentile_threshold` và `buffer_size`, các chunk sẽ có kích thước rất khác nhau.
- Việc tối ưu các tham số này là rất quan trọng để đảm bảo chunk phù hợp cho embedding:
    - Chunk quá lớn có thể làm giảm chất lượng embedding, tốn chi phí xử lý.
    - Chunk quá nhỏ có thể làm mất ngữ cảnh quan trọng.
- Nên thử nghiệm nhiều giá trị để tìm ra cấu hình tối ưu cho từng loại văn bản và mục đích sử dụng.